# Founder Diffusion Timing vs. Project Survival\n\nThis notebook demonstrates the founder-only **Truck-Factor Developer Departure (TFDD)** pipeline: it reconstructs per-repo commit history from mined `(commit, file)` rows, computes the **Degree-of-Authorship (DOA)** formula of Fritz et al. 2010 / Avelino et al. ICPC2016 to find each file's primary owner, detects the moment a repo drops to a single-developer Truck Factor and that developer then goes silent for 365+ days (a TFDD event), measures the **pre-departure authority-diffusion trajectory** (founder commit share and number of diffused owners in the 6-12 month window before departure), and classifies whether the project **survives** the following 18 months (recovers a non-founder Truck-Factor owner).\n\nIt also runs the iteration's new **Medappa-et-al.-style reconciliation model**: a static whole-history write-access ratio (`medappa_ratio`) plus a `timing_term` capturing how concentrated diffusion onset is near departure vs. spread through history, testing whether it is the *timing* of diffusion — not its mere presence — that predicts survival.\n\nThis demo runs the full pipeline on a small subset of repos (2 repos, ~1200 commit/file rows) from the original 34-repo founder-candidate corpus, small enough to finish in well under a minute while producing real founder-TFDD events end to end.

In [ ]:
import subprocess, sys\ndef _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])\n\n# statsmodels, loguru -- NOT pre-installed on Colab, always install\n_pip('statsmodels==0.14.6')\n_pip('loguru==0.7.2')\n\n# numpy, pandas, scipy -- pre-installed on Colab, install locally only (Colab's exact versions)\nif 'google.colab' not in sys.modules:\n    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3')

In [ ]:
from __future__ import annotations\n\nimport gc\nimport json\nimport math\nimport random\nimport sys\nimport time\nfrom collections import Counter, defaultdict\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime, timedelta, timezone\nfrom pathlib import Path\nfrom typing import Optional\n\nimport numpy as np\nimport pandas as pd\nimport statsmodels.api as sm\nfrom loguru import logger\nfrom scipy import stats\nfrom statsmodels.stats.outliers_influence import variance_inflation_factor\n\n# added for the notebook's results visualization cell\nimport matplotlib.pyplot as plt\n\nlogger.remove()\nlogger.add(sys.stdout, level=\"INFO\", format=\"{time:HH:mm:ss}|{level:<7}|{message}\")\n\nRNG_SEED = 20260821\nrandom.seed(RNG_SEED)\nnp.random.seed(RNG_SEED)

## Load the demo data\n\n`mini_demo_data.json` is a curated subset of the original mined `full_data_out.json`: the full per-`(commit, file)` row corpus for **two** of the 34 founder-candidate repos (`JustinSDK/JavaSE6Tutorial` and `Krupen/AutoplayVideos`, ~1200 rows total) — both of which are known to produce a genuine founder-only TFDD event under the original pipeline, so this small demo still exercises the full detection + survival + reconciliation logic end to end.\n\nThe loader tries the GitHub raw URL first (works once this artifact is pushed / on Colab), then falls back to the local file (works right now).

In [ ]:
GITHUB_DATA_URL = \"https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-24ffbe-pre-departure-bus-factor-diffusion/main/round-2/experiment-1/demo/mini_demo_data.json\"\nimport json, os\n\ndef load_data():\n    try:\n        import urllib.request\n        with urllib.request.urlopen(GITHUB_DATA_URL) as response:\n            return json.loads(response.read().decode())\n    except Exception: pass\n    if os.path.exists(\"mini_demo_data.json\"):\n        with open(\"mini_demo_data.json\") as f: return json.load(f)\n    raise FileNotFoundError(\"Could not load mini_demo_data.json\")

In [ ]:
data = load_data()\nprint(data[\"metadata\"])\nprint(\"n_examples in demo:\", len(data[\"datasets\"][0][\"examples\"]))

## Config\n\nAll tunable parameters from the original `method.py`, collected here. `N_BOOT` (bootstrap resamples) and the placebo-check resample count are the two knobs that dominate runtime, so they are set to small-but-nonzero values for this demo (the original values, used for the full 34-repo run, are commented alongside). Everything else (thresholds, window sizes) is reused **unchanged** from the validated iter1 pipeline, per the docstring's \"do not re-tune\" note.

In [ ]:
# --- constants reused verbatim from iter1 (do not re-tune) ---\nSILENCE_THRESHOLD_DAYS = 365\nTF_COVERAGE_THRESHOLD = 0.5\nPOST_TFDD_WINDOW_DAYS = 548  # 18 months\nPRE_WINDOW_FAR_DAYS = 365  # 12 months before TFDD\nPRE_WINDOW_NEAR_DAYS = 180  # 6 months before TFDD\nSNAPSHOT_EVERY_DAYS = 90\nSTRICT_FOUNDER_SHARE = 0.70  # re-verification threshold for founder-only strict criterion\nRELAXED_FOUNDER_SHARE = 0.50\nTARGET_N_STRICT = 40  # iter1 power-analysis target; 34-repo pool structurally caps below this\n\nAVELINO_REFERENCE_SURVIVAL_RATE = 0.41\n\n# --- runtime-dominating knobs: minimized for this small demo ---\nN_BOOT = 200  # original: 5000 (bootstrap CI resamples for matched-pairs / mean-diff CIs)\nN_PLACEBO_BOOT = 50  # original: 1000 (placebo-window empirical-null resamples)

## Step 0: reconstruct per-repo commit streams\n\nThe mined `full_data_out.json` (here, its `mini_demo_data.json` subset) stores one row per `(commit, file)` pair. This step groups rows back into per-repo, per-commit lists sorted by date — the same shape `git log --numstat` would have produced if the pipeline had cloned these repos live, but read from the already-mined dataset instead.

In [ ]:
def load_repo_commit_streams(raw: dict) -> dict[str, dict]:\n    examples = raw[\"datasets\"][0][\"examples\"]\n    logger.info(f\"[step0] loaded {len(examples)} (commit,file) rows\")\n\n    repos: dict[str, dict] = {}\n    for row in examples:\n        rid = str(row[\"metadata_repo_id\"])\n        rep = repos.setdefault(\n            rid,\n            {\n                \"full_name\": row[\"metadata_full_name\"],\n                \"license_key\": row.get(\"metadata_license\") or \"none\",\n                \"created_at\": datetime.fromisoformat(row[\"metadata_repo_created_at\"].replace(\"Z\", \"+00:00\")),\n                \"stars\": None,\n                \"forks\": None,\n                \"language\": None,\n                \"commits_by_sha\": {},\n                \"dominant_founder_first_window_share\": row.get(\"metadata_dominant_founder_share_first_window\"),\n                \"alias_ambiguous\": row.get(\"metadata_alias_ambiguous_repo\"),\n            },\n        )\n        try:\n            inp = json.loads(row[\"input\"])\n        except (json.JSONDecodeError, TypeError):\n            inp = {}\n        if rep[\"stars\"] is None:\n            rep[\"stars\"] = inp.get(\"repo_stars\", 0)\n            rep[\"forks\"] = inp.get(\"repo_forks\", 0)\n            rep[\"language\"] = inp.get(\"repo_primary_language\", \"unknown\")\n        sha = row[\"metadata_commit_sha\"]\n        c = rep[\"commits_by_sha\"].get(sha)\n        if c is None:\n            try:\n                dt = datetime.fromisoformat(row[\"metadata_commit_timestamp\"])\n            except ValueError:\n                continue\n            c = {\"hash\": sha, \"author_email\": row[\"metadata_author_alias_key\"], \"date\": dt, \"files\": []}\n            rep[\"commits_by_sha\"][sha] = c\n        added = inp.get(\"lines_added\", 0) or 0\n        removed = inp.get(\"lines_removed\", 0) or 0\n        c[\"files\"].append((inp.get(\"file_path\", \"?\"), added, removed))\n\n    for rep in repos.values():\n        commits = sorted(rep[\"commits_by_sha\"].values(), key=lambda c: c[\"date\"])\n        rep[\"commits\"] = commits\n        del rep[\"commits_by_sha\"]\n    logger.info(f\"[step0] grouped into {len(repos)} repos\")\n    return repos\n\n\nrepos = load_repo_commit_streams(data)\nn_repo_candidates = len(repos)\nfor rid, rep in repos.items():\n    print(rep[\"full_name\"], \"-\", len(rep[\"commits\"]), \"commits\")

## Degree-of-Authorship (DOA) and Truck-Factor\n\nReused verbatim from iter1: the Fritz et al. 2010 DOA formula as validated by Avelino et al. (ICPC 2016 / ESEM 2019), `DOA(dev, file, t) = 3.293 + 1.098*FA - 0.164*sqrt(AC) + 0.230*ln(1+DL)` where `FA` = first-author flag, `AC` = number of commits by that dev to that file, `DL` = deleted lines. The primary owner of a file is whoever has the highest DOA at a given cutoff date. The Truck Factor set is the smallest set of top owners whose files collectively cover >=50% of the repo.

In [ ]:
def doa_snapshot(commits: list[dict], cutoff: datetime) -> dict[tuple[str, str], float]:\n    file_dev_stats: dict[str, dict[str, dict]] = defaultdict(dict)\n    file_first_author: dict[str, str] = {}\n    for c in commits:\n        if c[\"date\"] > cutoff:\n            break\n        for path, added, deleted in c[\"files\"]:\n            if path not in file_first_author:\n                file_first_author[path] = c[\"author_email\"]\n            dev_stats = file_dev_stats[path]\n            s = dev_stats.setdefault(c[\"author_email\"], {\"ac\": 0, \"dl\": 0})\n            s[\"ac\"] += 1\n            s[\"dl\"] += deleted\n    doa: dict[tuple[str, str], float] = {}\n    for path, devs in file_dev_stats.items():\n        first_author = file_first_author[path]\n        for dev, s in devs.items():\n            fa = 1 if dev == first_author else 0\n            doa[(dev, path)] = 3.293 + 1.098 * fa - 0.164 * math.sqrt(s[\"ac\"]) + 0.230 * math.log(1 + s[\"dl\"])\n    return doa\n\n\ndef file_owners(doa: dict[tuple[str, str], float]) -> dict[str, tuple[str, float]]:\n    owner: dict[str, tuple[str, float]] = {}\n    for (dev, path), score in doa.items():\n        if path not in owner or score > owner[path][1]:\n            owner[path] = (dev, score)\n    return owner\n\n\ndef truck_factor_set(doa: dict[tuple[str, str], float]) -> list[str]:\n    owner = file_owners(doa)\n    total_files = len(owner)\n    if total_files == 0:\n        return []\n    owned_counts = Counter(dev for dev, _ in owner.values())\n    tf_set: list[str] = []\n    covered = 0\n    for dev, n in owned_counts.most_common():\n        tf_set.append(dev)\n        covered += n\n        if covered >= TF_COVERAGE_THRESHOLD * total_files:\n            break\n    return tf_set

## TFDD event detection\n\n`TFDDEvent` records one founder-only Truck-Factor Developer Departure: the moment a repo's Truck Factor drops to a single developer who then goes silent for `SILENCE_THRESHOLD_DAYS` (365 days). `detect_founder_tfdd` scans snapshots every `SNAPSHOT_EVERY_DAYS` (90 days) for the **strict** criterion (Truck Factor == 1); `detect_relaxed_tfdd` is a looser variant allowing a Truck Factor of 1 or 2.

In [ ]:
@dataclass\nclass TFDDEvent:\n    repo: str\n    founder: str\n    tfdd_date: datetime\n    repo_created_at: datetime\n    stars: int\n    forks: int\n    language: str\n    license_key: str\n    n_commits_total: int\n    tf_set_size_at_relaxed: int = 1\n    devs_at_tfdd: int = 0\n    commits_at_tfdd: int = 0\n    files_at_tfdd: int = 0\n    founder_share: float = float(\"nan\")\n    n_diffused_owners: int = 0\n    placebo_founder_share: float = float(\"nan\")\n    placebo_n_diffused_owners: int = 0\n    survived: Optional[bool] = None\n    grade: str = \"\"\n    censored: bool = False\n    devs_seen_up_to_tfdd: int = 0\n    # NEW (this iteration): reconciliation-test measurements\n    medappa_ratio: float = float(\"nan\")\n    timing_term: float = float(\"nan\")\n\n\ndef detect_founder_tfdd(commits: list[dict], snapshot_every_days: int = SNAPSHOT_EVERY_DAYS) -> Optional[tuple[datetime, str]]:\n    if len(commits) < 20:\n        return None\n    start = commits[0][\"date\"]\n    end = commits[-1][\"date\"]\n    last_active: dict[str, datetime] = {}\n    for c in commits:\n        e = c[\"author_email\"]\n        if e not in last_active or c[\"date\"] > last_active[e]:\n            last_active[e] = c[\"date\"]\n    cursor = start + timedelta(days=180)\n    while cursor <= end:\n        doa = doa_snapshot(commits, cursor)\n        tf_set = truck_factor_set(doa)\n        if len(tf_set) == 1:\n            founder = tf_set[0]\n            silence = (cursor - last_active.get(founder, start)).days\n            if silence >= SILENCE_THRESHOLD_DAYS:\n                tfdd_date = last_active[founder] + timedelta(days=SILENCE_THRESHOLD_DAYS)\n                return min(tfdd_date, cursor), founder\n        cursor += timedelta(days=snapshot_every_days)\n    return None\n\n\ndef detect_relaxed_tfdd(commits: list[dict], snapshot_every_days: int = SNAPSHOT_EVERY_DAYS) -> Optional[tuple[datetime, list[str]]]:\n    if len(commits) < 20:\n        return None\n    start = commits[0][\"date\"]\n    end = commits[-1][\"date\"]\n    last_active: dict[str, datetime] = {}\n    for c in commits:\n        e = c[\"author_email\"]\n        if e not in last_active or c[\"date\"] > last_active[e]:\n            last_active[e] = c[\"date\"]\n    cursor = start + timedelta(days=180)\n    while cursor <= end:\n        doa = doa_snapshot(commits, cursor)\n        tf_set = truck_factor_set(doa)\n        if 1 <= len(tf_set) <= 2 and all(\n            (cursor - last_active.get(d, start)).days >= SILENCE_THRESHOLD_DAYS for d in tf_set\n        ):\n            tfdd_date = max(last_active[d] for d in tf_set) + timedelta(days=SILENCE_THRESHOLD_DAYS)\n            return min(tfdd_date, cursor), tf_set\n        cursor += timedelta(days=snapshot_every_days)\n    return None

## Pre-departure diffusion window, placebo windows, and post-departure survival grading\n\n`window_metrics` measures `founder_share` (fraction of commits by the founder) and `n_diffused_owners` (distinct non-founder file owners) inside a given window. `sample_placebo_window` picks a random 180-day window elsewhere in the repo's history, for the placebo/shuffle null-distribution check later. `classify_grade`/`label_survival` determine the 18-month post-TFDD outcome (survived = recovers a non-founder Truck-Factor owner).

In [ ]:
def window_metrics(commits: list[dict], window_start: datetime, window_end: datetime, founder: str) -> tuple[float, int]:\n    window_commits = [c for c in commits if window_start <= c[\"date\"] < window_end]\n    if not window_commits:\n        return float(\"nan\"), 0\n    founder_commits = sum(1 for c in window_commits if c[\"author_email\"] == founder)\n    founder_share = founder_commits / len(window_commits)\n    doa_end = doa_snapshot(commits, window_end)\n    owner = file_owners(doa_end)\n    non_founder_owners = {o[0] for o in owner.values() if o[0] != founder}\n    return founder_share, len(non_founder_owners)\n\n\ndef sample_placebo_window(commits: list[dict], exclude_start: datetime, exclude_end: datetime) -> Optional[tuple[datetime, datetime]]:\n    start = commits[0][\"date\"]\n    end = commits[-1][\"date\"]\n    total_span_days = (end - start).days\n    if total_span_days < 800:\n        return None\n    for _ in range(20):\n        offset = random.uniform(0, total_span_days - 180)\n        w_start = start + timedelta(days=offset)\n        w_end = w_start + timedelta(days=180)\n        if w_end < exclude_start - timedelta(days=365) or w_start > exclude_end + timedelta(days=365):\n            return w_start, w_end\n    return None\n\n\ndef classify_grade(post_commits: list[dict], recovered_tf: list[str], founder: str) -> str:\n    if not post_commits:\n        return \"dead\"\n    n_devs = len({c[\"author_email\"] for c in post_commits})\n    n_commits = len(post_commits)\n    non_founder_tf = [d for d in recovered_tf if d != founder]\n    if non_founder_tf and n_commits >= 20 and n_devs >= 2:\n        return \"thriving\"\n    if n_commits >= 5:\n        return \"maintained\"\n    if n_commits >= 1:\n        return \"dormant\"\n    return \"dead\"\n\n\ndef label_survival(commits: list[dict], event: TFDDEvent, last_commit_date: datetime) -> None:\n    window_end = event.tfdd_date + timedelta(days=POST_TFDD_WINDOW_DAYS)\n    if last_commit_date < window_end:\n        event.censored = True\n    post = [c for c in commits if event.tfdd_date <= c[\"date\"] < window_end]\n    doa_post = doa_snapshot(commits, window_end)\n    recovered_tf = truck_factor_set(doa_post)\n    event.survived = bool(recovered_tf) and any(d != event.founder for d in recovered_tf)\n    event.grade = classify_grade(post, recovered_tf, event.founder)

## NEW (this iteration): Medappa-style reconciliation measurements\n\n`medappa_ratio` is a static, whole-pre-history write-access ratio (analog of Medappa et al.'s construct): the fraction of pre-TFDD developers who ever reached primary DOA ownership of at least one file. `timing_term` measures, among non-founder devs who own files AT the TFDD date, what fraction of their ownership *onset* dates fall inside the 6-12mo pre-departure window vs. earlier — testing whether diffusion is concentrated near departure or was already present in the repo's history.

In [ ]:
def compute_medappa_and_timing(\n    commits: list[dict], founder: str, tfdd_date: datetime, window_start: datetime, window_end: datetime\n) -> tuple[float, float]:\n    devs_before = {c[\"author_email\"] for c in commits if c[\"date\"] <= tfdd_date}\n    doa_tfdd = doa_snapshot(commits, tfdd_date)\n    owners_tfdd = file_owners(doa_tfdd)\n    ever_owners = {o[0] for o in owners_tfdd.values()}\n    medappa_ratio = len(ever_owners) / len(devs_before) if devs_before else float(\"nan\")\n\n    non_founder_owners_at_tfdd = {o[0] for o in owners_tfdd.values() if o[0] != founder}\n    if not non_founder_owners_at_tfdd:\n        return medappa_ratio, float(\"nan\")\n\n    start = commits[0][\"date\"]\n    onset_date: dict[str, datetime] = {}\n    cursor = start + timedelta(days=180)\n    remaining = set(non_founder_owners_at_tfdd)\n    while cursor <= tfdd_date and remaining:\n        doa = doa_snapshot(commits, cursor)\n        owners = file_owners(doa)\n        present = {o[0] for o in owners.values()}\n        newly_onset = remaining & present\n        for dev in newly_onset:\n            onset_date[dev] = cursor\n        remaining -= newly_onset\n        cursor += timedelta(days=SNAPSHOT_EVERY_DAYS)\n    for dev in remaining:  # never caught by a coarse snapshot before TFDD -> onset at TFDD itself\n        onset_date[dev] = tfdd_date\n\n    n_total = len(onset_date)\n    n_in_window = sum(1 for d in onset_date.values() if window_start <= d < window_end)\n    timing_term = n_in_window / n_total if n_total else float(\"nan\")\n    return medappa_ratio, timing_term

## Statistics helpers\n\nMatched-pairs construction (low- vs. high-diffusion repos matched on language/stars/forks/devs, with a relaxed same-stratum fallback), bootstrap survival-rate-ratio CIs, Benjamini-Hochberg FDR correction, Cohen's d, and bootstrap mean-difference CIs — all reused verbatim from iter1.

In [ ]:
def build_matched_pairs(df: pd.DataFrame, low_thresh: float = 0.50, hi_thresh: float = 0.80, n_diffused_min: int = 2):\n    lo = df[(df.founder_share < low_thresh) & (df.n_diffused_owners >= n_diffused_min)].copy()\n    hi = df[df.founder_share >= hi_thresh].copy()\n    pairs = []\n    used_hi = set()\n    for _, lrow in lo.iterrows():\n        best_idx, best_dist = None, float(\"inf\")\n        for hidx, hrow in hi.iterrows():\n            if hidx in used_hi or hrow.language != lrow.language:\n                continue\n            dist = (\n                (math.log1p(hrow.stars) - math.log1p(lrow.stars)) ** 2\n                + (math.log1p(hrow.forks) - math.log1p(lrow.forks)) ** 2\n                + (math.log1p(hrow.devs_at_tfdd) - math.log1p(lrow.devs_at_tfdd)) ** 2\n            )\n            if dist < best_dist:\n                best_dist, best_idx = dist, hidx\n        if best_idx is not None and best_dist < 4.0:\n            used_hi.add(best_idx)\n            pairs.append((lrow, hi.loc[best_idx]))\n    return pairs\n\n\ndef build_matched_pairs_relaxed(df: pd.DataFrame, low_thresh: float = 0.50, hi_thresh: float = 0.80, n_diffused_min: int = 2):\n    \"\"\"fallback_plan item (4): same-stratum-only matching, drop the exact language\n    requirement (kept as a regression covariate elsewhere instead).\"\"\"\n    lo = df[(df.founder_share < low_thresh) & (df.n_diffused_owners >= n_diffused_min)].copy()\n    hi = df[df.founder_share >= hi_thresh].copy()\n\n    def star_stratum(s: float) -> int:\n        return 0 if s < 1000 else (1 if s < 10000 else 2)\n\n    pairs = []\n    used_hi = set()\n    for _, lrow in lo.iterrows():\n        best_idx, best_dist = None, float(\"inf\")\n        for hidx, hrow in hi.iterrows():\n            if hidx in used_hi or star_stratum(hrow.stars) != star_stratum(lrow.stars):\n                continue\n            dist = (math.log1p(hrow.devs_at_tfdd) - math.log1p(lrow.devs_at_tfdd)) ** 2\n            if dist < best_dist:\n                best_dist, best_idx = dist, hidx\n        if best_idx is not None:\n            used_hi.add(best_idx)\n            pairs.append((lrow, hi.loc[best_idx]))\n    return pairs\n\n\ndef bootstrap_survival_rate_ratio(pairs: list[tuple[pd.Series, pd.Series]], n_boot: int = N_BOOT) -> tuple[float, tuple[float, float], Optional[str]]:\n    if not pairs:\n        return float(\"nan\"), (float(\"nan\"), float(\"nan\")), \"no matched pairs\"\n    lo_surv = np.array([1.0 if p[0].survived else 0.0 for p in pairs])\n    hi_surv = np.array([1.0 if p[1].survived else 0.0 for p in pairs])\n    n = len(pairs)\n    ratios = []\n    for _ in range(n_boot):\n        idx = np.random.randint(0, n, size=n)\n        lo_rate = lo_surv[idx].mean()\n        hi_rate = hi_surv[idx].mean()\n        if hi_rate == 0:\n            continue\n        ratios.append((lo_rate + 1e-6) / (hi_rate + 1e-6))\n    if not ratios:\n        degeneracy_note = (\n            f\"ALL {n_boot} bootstrap resamples had zero survivors in the high-diffusion group \"\n            f\"(hi_surv.mean()={hi_surv.mean():.3f} across the {n} matched pairs) -- the risk-ratio is \"\n            \"degenerate at this n, not computable, and NOT silently reported as a point estimate.\"\n        )\n        return float(\"nan\"), (float(\"nan\"), float(\"nan\")), degeneracy_note\n    ratios = np.array(ratios)\n    point = (lo_surv.mean() + 1e-6) / (hi_surv.mean() + 1e-6)\n    ci = (float(np.percentile(ratios, 2.5)), float(np.percentile(ratios, 97.5)))\n    return float(point), ci, None\n\n\ndef benjamini_hochberg(pvals: dict[str, float]) -> dict[str, float]:\n    items = sorted(pvals.items(), key=lambda kv: kv[1])\n    m = len(items)\n    adj = {}\n    prev = 1.0\n    for rank, (k, p) in enumerate(reversed(items), start=1):\n        r = m - rank + 1\n        val = min(prev, p * m / r)\n        adj[k] = val\n        prev = val\n    return adj\n\n\ndef cohens_d(a: np.ndarray, b: np.ndarray) -> float:\n    a, b = a[~np.isnan(a)], b[~np.isnan(b)]\n    if len(a) < 2 or len(b) < 2:\n        return float(\"nan\")\n    pooled_sd = math.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))\n    if pooled_sd == 0:\n        return float(\"nan\")\n    return float((a.mean() - b.mean()) / pooled_sd)\n\n\ndef bootstrap_ci_mean_diff(a: np.ndarray, b: np.ndarray, n_boot: int = N_BOOT) -> tuple[float, float]:\n    a, b = a[~np.isnan(a)], b[~np.isnan(b)]\n    if len(a) < 2 or len(b) < 2:\n        return float(\"nan\"), float(\"nan\")\n    diffs = np.empty(n_boot)\n    for i in range(n_boot):\n        ai = a[np.random.randint(0, len(a), len(a))]\n        bi = b[np.random.randint(0, len(b), len(b))]\n        diffs[i] = ai.mean() - bi.mean()\n    return float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))

## Per-repo processing\n\n`process_repo` runs the full strict + relaxed TFDD detection and event-construction pipeline for one repo. The original script parallelizes this across repos with a `ProcessPoolExecutor`; for this small demo (2 repos) we run it as a plain sequential loop below to keep the notebook simple — the per-repo logic itself is unchanged.

In [ ]:
def process_repo(repo_id: str, rep: dict) -> tuple[Optional[TFDDEvent], Optional[TFDDEvent], dict]:\n    full_name = rep[\"full_name\"]\n    commits = rep[\"commits\"]\n    diag = {\"repo\": full_name, \"stars\": rep[\"stars\"], \"language\": rep[\"language\"]}\n    if len(commits) < 20:\n        diag[\"status\"] = \"too_few_commits\"\n        return None, None, diag\n    n_devs_total = len({c[\"author_email\"] for c in commits})\n    if n_devs_total < 2:\n        diag[\"status\"] = \"single_dev_never_had_team\"\n        return None, None, diag\n    last_commit_date = commits[-1][\"date\"]\n\n    strict = detect_founder_tfdd(commits)\n    relaxed = detect_relaxed_tfdd(commits)\n    created_at = rep[\"created_at\"]\n    license_key = rep[\"license_key\"]\n\n    def make_event(tfdd_date: datetime, founder: str) -> Optional[TFDDEvent]:\n        window_start = tfdd_date - timedelta(days=PRE_WINDOW_FAR_DAYS)\n        window_end = tfdd_date - timedelta(days=PRE_WINDOW_NEAR_DAYS)\n        if window_start < commits[0][\"date\"]:\n            return None\n        founder_share, n_diffused = window_metrics(commits, window_start, window_end, founder)\n        if math.isnan(founder_share):\n            return None\n        doa_tfdd = doa_snapshot(commits, tfdd_date)\n        owners_tfdd = file_owners(doa_tfdd)\n        devs_before = {c[\"author_email\"] for c in commits if c[\"date\"] <= tfdd_date}\n        commits_before = [c for c in commits if c[\"date\"] <= tfdd_date]\n        ev = TFDDEvent(\n            repo=full_name,\n            founder=founder,\n            tfdd_date=tfdd_date,\n            repo_created_at=created_at,\n            stars=rep[\"stars\"] or 0,\n            forks=rep[\"forks\"] or 0,\n            language=rep[\"language\"] or \"unknown\",\n            license_key=license_key,\n            n_commits_total=len(commits),\n            devs_at_tfdd=len(devs_before),\n            commits_at_tfdd=len(commits_before),\n            files_at_tfdd=len(owners_tfdd),\n            founder_share=founder_share,\n            n_diffused_owners=n_diffused,\n            devs_seen_up_to_tfdd=len(devs_before),\n        )\n        placebo_window = sample_placebo_window(commits, window_start, window_end)\n        if placebo_window:\n            p_share, p_diff = window_metrics(commits, placebo_window[0], placebo_window[1], founder)\n            ev.placebo_founder_share = p_share\n            ev.placebo_n_diffused_owners = p_diff\n        label_survival(commits, ev, last_commit_date)\n        ev.medappa_ratio, ev.timing_term = compute_medappa_and_timing(commits, founder, tfdd_date, window_start, window_end)\n        return ev\n\n    strict_event = make_event(strict[0], strict[1]) if strict else None\n    relaxed_event = None\n    if relaxed:\n        r_date, r_set = relaxed\n        counts = Counter(c[\"author_email\"] for c in commits if c[\"author_email\"] in r_set)\n        dominant = counts.most_common(1)[0][0] if counts else r_set[0]\n        relaxed_event = make_event(r_date, dominant)\n        if relaxed_event is not None:\n            relaxed_event.tf_set_size_at_relaxed = len(r_set)\n\n    diag[\"status\"] = \"ok\"\n    diag[\"n_commits\"] = len(commits)\n    diag[\"n_devs\"] = n_devs_total\n    diag[\"strict_tfdd_found\"] = strict_event is not None\n    diag[\"relaxed_tfdd_found\"] = relaxed_event is not None\n    diag[\"dominant_founder_first_window_share\"] = rep.get(\"dominant_founder_first_window_share\")\n    return strict_event, relaxed_event, diag\n\n\ndef fit_logit(df_in: pd.DataFrame, cols: list[str], label: str) -> dict:\n    if df_in.empty or df_in[\"survived\"].nunique() < 2 or len(df_in) < len(cols) + 3:\n        return {\"status\": \"insufficient_data\", \"n\": int(len(df_in)), \"n_classes\": int(df_in[\"survived\"].nunique()) if not df_in.empty else 0}\n    X = df_in[cols].astype(float)\n    y = df_in[\"survived\"].astype(int)\n    X_const = sm.add_constant(X, has_constant=\"add\")\n    try:\n        model = sm.Logit(y, X_const).fit(disp=0, maxiter=200)\n    except Exception as e:\n        logger.warning(f\"[{label}] logit failed ({e}); dropping lowest-priority covariates in order\")\n        drop_order = [\"license_key\", \"contributor_count\", \"medappa_ratio:timing_term\", \"timing_term\"]\n        parsimonious = [c for c in cols if c not in drop_order]\n        if not parsimonious or set(parsimonious) == set(cols):\n            return {\"status\": f\"failed:{e}\", \"n\": int(len(df_in))}\n        return fit_logit(df_in, parsimonious, label + \"_parsimonious\")\n    std_X = (X - X.mean()) / X.std(ddof=0).replace(0, 1)\n    std_X_const = sm.add_constant(std_X, has_constant=\"add\")\n    try:\n        std_model = sm.Logit(y, std_X_const).fit(disp=0, maxiter=200)\n        std_effects = std_model.params.drop(\"const\").to_dict()\n    except Exception:\n        std_effects = {}\n    return {\n        \"status\": \"ok\",\n        \"n\": int(len(df_in)),\n        \"covariates\": cols,\n        \"coefs\": model.params.to_dict(),\n        \"pvalues\": model.pvalues.to_dict(),\n        \"pvalues_bh\": benjamini_hochberg(model.pvalues.drop(\"const\").to_dict()),\n        \"standardized_effect_sizes\": std_effects,\n        \"pseudo_r2\": float(model.prsquared),\n        \"converged\": bool(model.mle_retvals.get(\"converged\", True)),\n    }\n\n\ndef compute_vif(df_in: pd.DataFrame, cols: list[str]) -> dict:\n    if df_in.empty or len(df_in) < len(cols) + 2:\n        return {\"status\": \"insufficient_data\"}\n    X = sm.add_constant(df_in[cols].astype(float), has_constant=\"add\")\n    vifs = {}\n    for i, c in enumerate(X.columns):\n        if c == \"const\":\n            continue\n        try:\n            vifs[c] = float(variance_inflation_factor(X.values, i))\n        except (ZeroDivisionError, np.linalg.LinAlgError, ValueError):\n            vifs[c] = float(\"nan\")\n    return vifs

## Run the pipeline over all demo repos

In [ ]:
t0 = time.time()\nlogger.info(f\"=== STEP 1-3: DOA/TF/TFDD pipeline, {n_repo_candidates} repos (sequential in this demo) ===\")\nstrict_events: list[dict] = []\nrelaxed_events: list[dict] = []\ndiagnostics: list[dict] = []\nfor i, (repo_id, rep) in enumerate(repos.items(), start=1):\n    try:\n        s_ev, r_ev, diag = process_repo(repo_id, rep)\n    except Exception as e:\n        logger.error(f\"[process_repo] {rep.get('full_name')} failed: {e}\")\n        s_ev, r_ev, diag = None, None, {\"repo\": rep.get(\"full_name\"), \"status\": f\"exception:{e}\"}\n    diagnostics.append(diag)\n    if s_ev is not None:\n        strict_events.append(asdict(s_ev))\n    if r_ev is not None:\n        relaxed_events.append(asdict(r_ev))\n    logger.info(f\"[step1-3] ({i}/{len(repos)}) {rep['full_name']}: {diag.get('status')}\")\n\nlogger.info(f\"=== Finished: {len(repos)} repos, {len(strict_events)} strict events, {len(relaxed_events)} relaxed events, {time.time()-t0:.1f}s ===\")\ndiag_df = pd.DataFrame(diagnostics)\ndiag_df

## Survival rates and analysis dataframes\n\nParse the raw event dicts back into `TFDDEvent` objects, compute unconditioned survival rates for the strict and relaxed criteria, and build the pandas analysis dataframes (log-transformed snapshot covariates, dropping censored events).

In [ ]:
n_strict, n_relaxed = len(strict_events), len(relaxed_events)\n\ndef parse_events(raw_events: list[dict]) -> list[TFDDEvent]:\n    out = []\n    for d in raw_events:\n        d = dict(d)\n        d[\"tfdd_date\"] = pd.to_datetime(d[\"tfdd_date\"], utc=True).to_pydatetime()\n        d[\"repo_created_at\"] = pd.to_datetime(d[\"repo_created_at\"], utc=True).to_pydatetime()\n        out.append(TFDDEvent(**d))\n    return out\n\nstrict_ev_objs = parse_events(strict_events)\nrelaxed_ev_objs = parse_events(relaxed_events)\n\ndef rate_summary(events: list[TFDDEvent]) -> dict:\n    uncensored = [e for e in events if not e.censored]\n    if not uncensored:\n        return {\"n_events\": len(events), \"n_uncensored\": 0, \"survival_rate\": None, \"n_censored_excluded\": len(events)}\n    surv = np.array([1.0 if e.survived else 0.0 for e in uncensored])\n    return {\n        \"n_events\": len(events),\n        \"n_uncensored\": len(uncensored),\n        \"n_censored_excluded\": len(events) - len(uncensored),\n        \"survival_rate\": float(surv.mean()),\n        \"survival_rate_se\": float(surv.std(ddof=1) / math.sqrt(len(surv))) if len(surv) > 1 else None,\n    }\n\nstrict_rate = rate_summary(strict_ev_objs)\nrelaxed_rate = rate_summary(relaxed_ev_objs)\nlogger.info(f\"[step6] strict founder-only TFDD survival: {strict_rate}\")\nlogger.info(f\"[step6] relaxed TF<=2 TFDD survival: {relaxed_rate}\")\n\ndef events_to_df(events: list[TFDDEvent]) -> pd.DataFrame:\n    rows = [asdict(e) for e in events if not e.censored]\n    if not rows:\n        return pd.DataFrame()\n    df = pd.DataFrame(rows)\n    df[\"log_stars\"] = np.log1p(df[\"stars\"])\n    df[\"log_forks\"] = np.log1p(df[\"forks\"])\n    df[\"log_devs_at_tfdd\"] = np.log1p(df[\"devs_at_tfdd\"])\n    df = df.dropna(subset=[\"founder_share\", \"n_diffused_owners\", \"log_stars\", \"log_forks\", \"devs_at_tfdd\"])\n    return df\n\ndf = events_to_df(strict_ev_objs)\ndf_relaxed = events_to_df(relaxed_ev_objs)\nprint(\"strict analysis rows:\", len(df), \"| relaxed analysis rows:\", len(df_relaxed))\ndf

## Primary statistical battery\n\nBH-corrected logistic regression of survival on `our_method` (diffusion trajectory + snapshot covariates) vs. a `baseline` (snapshot covariates only), Cohen's d / bootstrap CIs on individual covariates, matched-pairs risk-ratio analysis, and Mann-Whitney tests. With only a handful of demo events these tests will mostly report `insufficient_data` — exactly as the original script does when it lacks statistical power — rather than silently fabricating a result.

In [ ]:
results: dict = {\n    \"n_repos_input\": n_repo_candidates,\n    \"n_founder_candidates\": n_repo_candidates,\n    \"n_strict_tfdd\": n_strict,\n    \"n_relaxed_tfdd\": n_relaxed,\n    \"target_n\": TARGET_N_STRICT,\n    \"strict_unconditioned_survival\": strict_rate,\n    \"relaxed_unconditioned_survival\": relaxed_rate,\n    \"avelino_et_al_reference_survival_rate\": AVELINO_REFERENCE_SURVIVAL_RATE,\n    \"n_analysis_rows_strict\": int(len(df)),\n    \"n_analysis_rows_relaxed\": int(len(df_relaxed)),\n}\n\n# ---- primary battery ----\nour_cols = [\"founder_share\", \"n_diffused_owners\", \"log_stars\", \"log_forks\", \"log_devs_at_tfdd\"]\nbaseline_cols = [\"log_stars\", \"log_forks\", \"log_devs_at_tfdd\"]\nresults[\"primary_regression\"] = {\n    \"our_method\": fit_logit(df, our_cols, \"our_method\"),\n    \"baseline_snapshot_only\": fit_logit(df, baseline_cols, \"baseline\"),\n}\nif not df.empty and df[\"survived\"].nunique() == 2:\n    surv_mask = df[\"survived\"].astype(bool)\n    cov_effects = {}\n    for col in [\"devs_at_tfdd\", \"commits_at_tfdd\", \"files_at_tfdd\", \"founder_share\", \"n_diffused_owners\"]:\n        a = df.loc[surv_mask, col].to_numpy(dtype=float)\n        b = df.loc[~surv_mask, col].to_numpy(dtype=float)\n        cov_effects[col] = {\"cohens_d\": cohens_d(a, b), \"bootstrap_ci95_mean_diff\": list(bootstrap_ci_mean_diff(a, b, n_boot=N_BOOT))}\n    results[\"primary_regression\"][\"snapshot_covariate_effect_sizes\"] = cov_effects\nelse:\n    results[\"primary_regression\"][\"snapshot_covariate_effect_sizes\"] = {\"status\": \"insufficient_class_variation\"}\n\nmatched_pairs_result = {\"n_pairs\": 0}\nif len(df) >= 6:\n    pairs = build_matched_pairs(df)\n    if pairs:\n        risk_ratio, ci95, degeneracy_note = bootstrap_survival_rate_ratio(pairs, n_boot=N_BOOT)\n        matched_pairs_result = {\n            \"n_pairs\": len(pairs),\n            \"matching\": \"strict (exact language + star/fork/devs distance)\",\n            \"risk_ratio_low_vs_high_diffusion\": risk_ratio,\n            \"risk_ratio_ci95\": list(ci95),\n            \"note\": degeneracy_note or \"risk_ratio = P(survival|low diffusion) / P(survival|high diffusion); >1 => concentrated founder survives MORE\",\n        }\n    else:\n        relaxed_pairs = build_matched_pairs_relaxed(df)\n        if relaxed_pairs:\n            risk_ratio, ci95, degeneracy_note = bootstrap_survival_rate_ratio(relaxed_pairs, n_boot=N_BOOT)\n            matched_pairs_result = {\n                \"n_pairs\": len(relaxed_pairs),\n                \"matching\": \"RELAXED (fallback_plan item 4): same star-stratum only, language dropped as exact match\",\n                \"risk_ratio_low_vs_high_diffusion\": risk_ratio,\n                \"risk_ratio_ci95\": list(ci95),\n                \"note\": degeneracy_note,\n            }\n        else:\n            matched_pairs_result[\"note\"] = \"ZERO eligible pairs even under relaxed same-stratum matching -- reporting explicitly rather than omitting\"\nelse:\n    matched_pairs_result[\"note\"] = \"insufficient events for matched-pairs analysis (need >=6)\"\nresults[\"matched_pairs\"] = matched_pairs_result\n\nif not df.empty and df[\"survived\"].nunique() == 2:\n    surv_mask = df[\"survived\"].astype(bool)\n    mw = {}\n    for col in [\"founder_share\", \"n_diffused_owners\"]:\n        res = stats.mannwhitneyu(df.loc[surv_mask, col], df.loc[~surv_mask, col], alternative=\"two-sided\")\n        mw[col] = {\"u_stat\": float(res.statistic), \"p\": float(res.pvalue)}\n    results[\"mann_whitney\"] = mw\nelse:\n    results[\"mann_whitney\"] = {\"status\": \"insufficient_class_variation\"}\n\nresults[\"primary_regression\"]

## Placebo / shuffle check\n\nRe-fits the regression using the randomly-placed placebo windows instead of the true pre-departure window, then bootstraps (`N_PLACEBO_BOOT` resamples here, 1000 in the original full run) an empirical null distribution for the founder_share coefficient — the true window's coefficient is compared against this null to get an empirical p-value.

In [ ]:
placebo_df = df.dropna(subset=[\"placebo_founder_share\", \"placebo_n_diffused_owners\"]).copy()\nplacebo_cols = [\"placebo_founder_share\", \"placebo_n_diffused_owners\", \"log_stars\", \"log_forks\", \"log_devs_at_tfdd\"]\nplacebo_reg = fit_logit(placebo_df, placebo_cols, \"placebo\") if len(placebo_df) >= 8 else {\"status\": \"insufficient_data\", \"n\": int(len(placebo_df))}\ntrue_coef = results[\"primary_regression\"][\"our_method\"].get(\"coefs\", {}).get(\"founder_share\") if results[\"primary_regression\"][\"our_method\"].get(\"status\") == \"ok\" else None\nnull_coefs = []\nif len(placebo_df) >= 8:\n    for _ in range(N_PLACEBO_BOOT):\n        boot_idx = np.random.randint(0, len(placebo_df), len(placebo_df))\n        boot_df = placebo_df.iloc[boot_idx]\n        r = fit_logit(boot_df, placebo_cols, \"placebo_boot\")\n        if r.get(\"status\") == \"ok\":\n            null_coefs.append(r[\"coefs\"].get(\"placebo_founder_share\", np.nan))\nnull_coefs = np.array([c for c in null_coefs if not np.isnan(c)])\nempirical_p = None\nif true_coef is not None and len(null_coefs) > 10:\n    empirical_p = float((np.abs(null_coefs) >= abs(true_coef)).mean())\nresults[\"placebo_check\"] = {\n    \"n_events_with_placebo_window\": int(len(placebo_df)),\n    \"regression_placebo_window\": placebo_reg,\n    \"true_window_founder_share_coef\": true_coef,\n    \"null_distribution_summary\": {\n        \"n\": int(len(null_coefs)),\n        \"mean\": float(null_coefs.mean()) if len(null_coefs) else None,\n        \"std\": float(null_coefs.std()) if len(null_coefs) else None,\n    },\n    \"empirical_p\": empirical_p,\n}\n\nif len(df_relaxed) >= 6 and df_relaxed[\"survived\"].nunique() == 2:\n    results[\"relaxed_sensitivity_regression\"] = fit_logit(df_relaxed, our_cols, \"relaxed_our_method\")\nelse:\n    results[\"relaxed_sensitivity_regression\"] = {\"status\": \"insufficient_data\", \"n\": int(len(df_relaxed))}\n\nresults[\"placebo_check\"]

## Medappa reconciliation joint model\n\nThe new test for this iteration: does `timing_term` moderate or flip the sign of `medappa_ratio`'s association with survival? Reports univariate associations, a VIF collinearity check between `medappa_ratio` and `founder_share`, and the joint logistic model with the `medappa_ratio x timing_term` interaction.

In [ ]:
recon_df = df.dropna(subset=[\"medappa_ratio\", \"timing_term\"]).copy()\nrecon_df[\"medappa_x_timing\"] = recon_df[\"medappa_ratio\"] * recon_df[\"timing_term\"]\nreconciliation: dict = {\"n_events\": int(len(recon_df))}\n\nif len(recon_df) >= 6 and recon_df[\"survived\"].nunique() == 2:\n    univariate = {}\n    surv_mask = recon_df[\"survived\"].astype(bool)\n    for col in [\"medappa_ratio\", \"timing_term\", \"founder_share\"]:\n        a = recon_df.loc[surv_mask, col].to_numpy(dtype=float)\n        b = recon_df.loc[~surv_mask, col].to_numpy(dtype=float)\n        res = stats.mannwhitneyu(a, b, alternative=\"two-sided\") if len(a) >= 1 and len(b) >= 1 else None\n        univariate[col] = {\n            \"cohens_d\": cohens_d(a, b),\n            \"mannwhitney_p\": float(res.pvalue) if res is not None else None,\n            \"mean_survived\": float(np.nanmean(a)) if len(a) else None,\n            \"mean_not_survived\": float(np.nanmean(b)) if len(b) else None,\n        }\n    reconciliation[\"univariate_associations\"] = univariate\n\n    vif = compute_vif(recon_df, [\"medappa_ratio\", \"founder_share\"])\n    reconciliation[\"vif_medappa_vs_founder_share\"] = vif\n    high_vif = isinstance(vif, dict) and any(isinstance(v, float) and v > 10 for v in vif.values())\n    reconciliation[\"multicollinearity_flag\"] = bool(high_vif)\n\n    base_reconcile_cols = [\"founder_share\", \"medappa_ratio\", \"timing_term\", \"medappa_x_timing\", \"log_stars\", \"log_forks\", \"log_devs_at_tfdd\"]\n    joint = fit_logit(recon_df, base_reconcile_cols, \"reconciliation_joint\")\n    reconciliation[\"joint_model\"] = joint\n\n    medappa_only = fit_logit(recon_df, [\"medappa_ratio\", \"log_stars\", \"log_forks\", \"log_devs_at_tfdd\"], \"medappa_alone\")\n    reconciliation[\"medappa_alone_model\"] = medappa_only\n    medappa_sign = None\n    if medappa_only.get(\"status\") == \"ok\":\n        medappa_sign = \"negative\" if medappa_only[\"coefs\"].get(\"medappa_ratio\", 0) < 0 else \"positive\"\n    reconciliation[\"medappa_alone_sign\"] = medappa_sign\n    reconciliation[\"replicates_medappa_negative_direction\"] = (medappa_sign == \"negative\")\n\n    interp_parts = []\n    if high_vif:\n        interp_parts.append(\n            \"VIF>10 between medappa_ratio and founder_share: the two constructs are NOT cleanly separable \"\n            f\"at n={len(recon_df)}; the joint model's medappa/founder_share coefficients should not be \"\n            \"interpreted independently. Reporting separate univariate associations as the primary evidence instead.\"\n        )\n    if joint.get(\"status\") == \"ok\":\n        interaction_coef = joint[\"coefs\"].get(\"medappa_x_timing\")\n        timing_coef = joint[\"coefs\"].get(\"timing_term\")\n        interp_parts.append(\n            f\"Joint model converged (n={joint['n']}); medappa_ratio coef={joint['coefs'].get('medappa_ratio'):.3f}, \"\n            f\"timing_term coef={timing_coef:.3f}, interaction coef={interaction_coef:.3f}. \"\n            + (\"Timing/interaction term MODERATES or FLIPS the medappa_ratio-alone sign, consistent with the \"\n               \"timing-not-presence reconciliation hypothesis.\"\n               if (medappa_sign == \"negative\" and interaction_coef is not None and (interaction_coef > 0) != (joint[\"coefs\"].get(\"medappa_ratio\", 0) > 0))\n               else \"Timing/interaction term does NOT clearly flip the medappa_ratio-alone sign at this n; \"\n                    \"underpowered to distinguish timing-driven from presence-driven mechanisms.\")\n        )\n    else:\n        interp_parts.append(f\"Joint model did not converge cleanly ({joint.get('status')}); falling back to separate univariate associations per fallback_plan item (5).\")\n    reconciliation[\"interpretation\"] = \" \".join(interp_parts)\nelse:\n    reconciliation[\"status\"] = \"insufficient_data_or_class_variation\"\n    reconciliation[\"note\"] = f\"n={len(recon_df)} events with complete medappa_ratio/timing_term measurements; need >=6 with both survival classes present\"\nresults[\"reconciliation\"] = reconciliation\nresults[\"runtime_seconds\"] = time.time() - t0\nreconciliation

## Results summary\n\nA readable summary table of the per-event raw features/outcome, plus a bar chart of pre-departure founder commit share for each detected strict TFDD event, colored by whether the project survived the following 18 months.

In [ ]:
print(f\"n_repos_input={n_repo_candidates}  n_strict_tfdd={n_strict}  n_relaxed_tfdd={n_relaxed}  target_n={TARGET_N_STRICT}\")\nprint(f\"strict_unconditioned_survival={strict_rate}\")\nprint(f\"avelino_et_al_reference_survival_rate={AVELINO_REFERENCE_SURVIVAL_RATE}\")\nprint(f\"runtime_seconds={results['runtime_seconds']:.2f}\")\n\nsummary_cols = [\"repo\", \"founder\", \"founder_share\", \"n_diffused_owners\", \"medappa_ratio\", \"timing_term\", \"stars\", \"devs_at_tfdd\", \"survived\", \"grade\", \"censored\"]\nif not df.empty:\n    display_df = df[summary_cols].copy()\n    print(display_df.to_string(index=False))\nelse:\n    display_df = pd.DataFrame(columns=summary_cols)\n    print(\"No uncensored strict TFDD events in this demo subset.\")\n\nfig, ax = plt.subplots(figsize=(7, 4))\nif not display_df.empty:\n    colors = [\"#2a9d8f\" if s else \"#e76f51\" for s in display_df[\"survived\"]]\n    labels = [f\"{r.split('/')[-1]}\\n{d.date()}\" for r, d in zip(display_df[\"repo\"], df[\"tfdd_date\"])]\n    ax.bar(range(len(display_df)), display_df[\"founder_share\"], color=colors)\n    ax.set_xticks(range(len(display_df)))\n    ax.set_xticklabels(labels, rotation=0, fontsize=8)\n    ax.axhline(AVELINO_REFERENCE_SURVIVAL_RATE, color=\"gray\", linestyle=\"--\", linewidth=1, label=f\"Avelino et al. reference survival rate ({AVELINO_REFERENCE_SURVIVAL_RATE})\")\n    ax.set_ylabel(\"pre-departure founder_share (12-6mo window)\")\n    ax.set_title(\"Founder commit share before TFDD, by event outcome\")\n    ax.legend(handles=[\n        plt.Rectangle((0, 0), 1, 1, color=\"#2a9d8f\", label=\"survived\"),\n        plt.Rectangle((0, 0), 1, 1, color=\"#e76f51\", label=\"did not survive\"),\n    ], fontsize=8)\nelse:\n    ax.text(0.5, 0.5, \"no strict TFDD events detected in this demo subset\", ha=\"center\", va=\"center\")\nplt.tight_layout()\nplt.show()